# Module 1: Caries YOLO training in Google Colab

This notebook accepts a Roboflow/Ultralytics ZIP such as `Caries.v12i.yolo26.zip`, trains on a Colab GPU, tests an X-ray image, and downloads `best.pt` for the local Module 1 API.

Before training: Runtime > Change runtime type > T4 GPU.

In [ ]:
!pip -q install ultralytics pyyaml
import shutil
from pathlib import Path
import yaml
from google.colab import files

print('GPU check:')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise ValueError('Upload the YOLO dataset ZIP file.')
zip_path = Path('/content') / zip_names[0]
dataset_root = Path('/content/caries_dataset')
if dataset_root.exists():
    shutil.rmtree(dataset_root)
dataset_root.mkdir()
!unzip -q "{zip_path}" -d "{dataset_root}"

yaml_files = list(dataset_root.rglob('data.yaml')) + list(dataset_root.rglob('data.yml'))
if not yaml_files:
    raise FileNotFoundError('No data.yaml or data.yml found inside the ZIP.')
data_yaml = yaml_files[0]
data = yaml.safe_load(data_yaml.read_text())
print('Using:', data_yaml)
print(yaml.safe_dump(data, sort_keys=False))
if not data.get('names'):
    raise ValueError('data.yaml must define class names under names.')

## Train

`yolo26n.pt` is a small starting model. Increase to `yolo26s.pt` or `yolo26m.pt` if the dataset and GPU allow it.

In [ ]:
from ultralytics import YOLO

MODEL_NAME = 'yolo26n.pt'
EPOCHS = 50
IMAGE_SIZE = 640

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=-1,
    patience=15,
    project='/content/caries_runs',
    name='module1_caries',
    device=0,
)

best_weights = Path('/content/caries_runs/module1_caries/weights/best.pt')
if not best_weights.exists():
    raise FileNotFoundError(f'Training finished but {best_weights} was not created.')
print('Best weights:', best_weights)

In [ ]:
# Optional: upload one panoramic X-ray and inspect predictions.
test_upload = files.upload()
test_images = [Path('/content') / name for name in test_upload if Path(name).suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp'}]
if test_images:
    trained_model = YOLO(str(best_weights))
    prediction = trained_model.predict(source=str(test_images[0]), conf=0.25, save=True, project='/content/caries_predictions', name='xray')
    print('Prediction image:', prediction[0].save_dir / test_images[0].name)
else:
    print('No test image uploaded; skip prediction preview.')

In [ ]:
# Download the artifact expected by services/module1-caries-enamel.
deployment_dir = Path('/content/module1_caries_model')
if deployment_dir.exists():
    shutil.rmtree(deployment_dir)
deployment_dir.mkdir()
shutil.copy2(best_weights, deployment_dir / 'best.pt')
shutil.make_archive('/content/module1_caries_model', 'zip', deployment_dir)
files.download('/content/module1_caries_model.zip')

## Use the weights locally

1. Extract `module1_caries_model.zip`.
2. Copy `best.pt` to `services/module1-caries-enamel/models_store/caries_detector/best.pt`.
3. Install the repository requirements and start Module 1 on port 8001.
4. Upload an X-ray in the frontend, or call `POST http://localhost:8001/predict`.

The detector reads YOLO class names from `data.yaml`. If a name is an FDI number, that number is returned; otherwise the API keeps its legacy response shape and reports class index + 1 in `fdi_number`.